In [ ]:
import torch
print(torch.cuda.is_available())


In [ ]:
!pip install easyocr pillow opencv-python-headless numpy

In [ ]:
import easyocr
import re
from PIL import Image
import numpy as np

def parse_chat_by_x_column_to_log(image_path, reader):
    """
    X좌표(들여쓰기)를 분석하여 '메인 텍스트 열'과 '타임스탬프 열'을
    구분하고, 이를 기반으로 "로그" 형식으로 파싱합니다.
    (기존 코드 기반, 출력 형식만 수정)
    """

    # 0. 이미지 크기 및 중앙선 계산
    try:
        pil_img = Image.open(image_path)
        img_width, img_height = pil_img.size
        center_x = img_width / 2
        print(f"--- 이미지 너비: {img_width}px, 높이: {img_height}px, 중앙 X: {center_x:.0f}px ---")
    except Exception as e:
        print(f"이미지 크기 읽기 실패: {e}")
        return

    # 1. OCR 실행 (좌표 포함)
    results = reader.readtext(image_path, paragraph=False)

    # 2. 텍스트 블록 정렬 및 처리 (왼쪽/오른쪽 구분)
    left_blocks = []  # (y_top, x_left, text)
    right_blocks = [] # (y_top, x_left, text)

    # 상단 상태바 필터링 기준 (전체 높이의 5%)
    top_filter_threshold = img_height * 0.05

    for (bbox, text, prob) in results:
        y_top = bbox[0][1]
        x_left = bbox[0][0]
        x_center = (bbox[0][0] + bbox[1][0]) / 2

        # 1. 상단 상태바 영역(상위 5%) 무시
        if y_top < top_filter_threshold:
            continue
        # 2. 안 읽은 수 무시
        if len(text) < 3 and text.isdigit():
            continue

        if x_center > center_x:
            right_blocks.append((y_top, x_left, text))
        else:
            left_blocks.append((y_top, x_left, text))

    # 3. [핵심 로직] 왼쪽 블록의 X좌표 통계 분석
    if not left_blocks and not right_blocks:
        print("텍스트 블록을 찾을 수 없습니다.")
        return

    main_text_column_x = 0
    column_threshold = img_width * 0.1
    if left_blocks: # 왼쪽 블록이 있을 때만 중앙값 계산
        x_coords = [x for (y, x, text) in left_blocks]
        if x_coords: # x_coords가 비어있지 않은지 확인
            main_text_column_x = np.median(x_coords)

    print(f"--- 1단계: 메인 텍스트 열(x) 중앙값: {main_text_column_x:.0f}px ---")


    # 4. 모든 블록을 y_top 기준으로 재정렬
    all_blocks = left_blocks + right_blocks
    all_blocks.sort(key=lambda b: b[0])

    # 5. 정렬된 블록을 순회하며 대화 재구성
    chat_log = []
    current_speaker = None

    # [수정] 타임스탬프 저장을 위한 변수와 정규식 추가
    current_timestamp = "시간 불명"
    time_regex = re.compile(r"^(오전|오후) \d{1,2}:\d{2}$|^\d{1,2}:\d{2}$")

    print("\n--- 2단계: 텍스트 추출 및 파싱 결과 ---")

    for (y, x, text) in all_blocks:

        # 5-1. '나 (Me)'의 메시지 (오른쪽)
        if x > center_x:
            current_speaker = "나 (Me)"

            # [수정] '나'의 타임스탬프인지 확인
            if time_regex.match(text):
                current_timestamp = text
                continue # 타임스탬프 자체는 출력하지 않음

            # [수정] 로그 형식으로 출력
            chat_log.append(f"{current_timestamp}, {current_speaker} : {text}")
            continue

        # 5-2. '왼쪽' 블록 분석
        # main_text_column_x가 0일 경우(왼쪽 블록이 없을 때)를 대비해
        is_outside_main_column = (main_text_column_x > 0 and abs(x - main_text_column_x) > column_threshold)

        if is_outside_main_column:
            # [수정] 타임스탬프 열이면, 텍스트를 저장
            if time_regex.match(text):
                current_timestamp = text

            # (시스템 메시지 등도 '턴 종료' 신호로 간주)
            current_speaker = None # 발화자 초기화
            continue

        else:
            # '메인 열'에 속한 텍스트 (이름 또는 메시지)
            if current_speaker is None:
                current_speaker = text
                # [수정] "---" 헤더는 출력하지 않음

            elif current_speaker == "나 (Me)":
                current_speaker = text
                # [수정] "---" 헤더는 출력하지 않음

            else:
                # [수정] 로그 형식으로 출력
                chat_log.append(f"{current_timestamp}, {current_speaker} : {text}")

    # 6. 최종 결과 출력
    for line in chat_log:
        print(line)

# --- 실행 ---
try:
    reader = easyocr.Reader(['ko', 'en'], gpu=True)
    print("EasyOCR 리더기 로드 완료.")

    image_file = "image.jpeg"
    parse_chat_by_x_column_to_log(image_file, reader)

    print("\n" + "="*30 + "\n")

    image_file = "image.png"
    parse_chat_by_x_column_to_log(image_file, reader)

except Exception as e:
    print(f"오류 발생: {e}")


In [ ]:
import easyocr
import re
from PIL import Image
import numpy as np

def parse_chat_by_x_boundary(image_path, reader):
    """
    X좌표 중앙값을 '결정 경계'로 사용하여
    '이름'과 '메시지'의 미세한 들여쓰기 차이를 감지하고 파싱합니다.
    """

    # 0. 이미지 크기 및 중앙선 계산
    try:
        pil_img = Image.open(image_path)
        img_width, img_height = pil_img.size
        center_x = img_width / 2
        print(f"--- 이미지 너비: {img_width}px, 높이: {img_height}px, 중앙 X: {center_x:.0f}px ---")
    except Exception as e:
        print(f"이미지 크기 읽기 실패: {e}")
        return

    # 1. OCR 실행
    results = reader.readtext(image_path, paragraph=False)

    # 2. 텍스트 블록 정렬 및 처리 (왼쪽/오른쪽 구분)
    left_blocks = []  # (y_top, x_left, text)
    right_blocks = [] # (y_top, x_left, text)

    top_filter_threshold = img_height * 0.05

    for (bbox, text, prob) in results:
        y_top = bbox[0][1]
        x_left = bbox[0][0]
        x_center = (bbox[0][0] + bbox[1][0]) / 2

        if y_top < top_filter_threshold: continue
        if len(text) < 3 and text.isdigit(): continue

        if x_center > center_x:
            right_blocks.append((y_top, x_left, text))
        else:
            left_blocks.append((y_top, x_left, text))

    # 3. 왼쪽 블록의 X좌표 통계 분석
    if not left_blocks:
        print("왼쪽 텍스트 블록을 찾을 수 없습니다.")
        # '나'의 대화만 있을 수 있으므로 계속 진행

    main_text_column_x = 0
    column_threshold = img_width * 0.1
    if left_blocks:
        x_coords = [x for (y, x, text) in left_blocks]
        main_text_column_x = np.median(x_coords)
        print(f"--- 1단계: 메인 텍스트 열(x) 중앙값(경계): {main_text_column_x:.0f}px ---")
    else:
        print("--- 1단계: 왼쪽 텍스트 블록 없음 ---")


    # 4. 모든 블록을 y_top 기준으로 재정렬
    all_blocks = left_blocks + right_blocks
    all_blocks.sort(key=lambda b: b[0])

    # 5. 정렬된 블록을 순회하며 대화 재구성
    chat_log = []
    current_speaker = "알 수 없음" # '나'가 먼저 말할 수도 있으므로 초기값 설정
    current_timestamp = "시간 불명"

    time_regex = re.compile(r"^(오전|오후) \d{1,2}:\d{2}$|^\d{1,2}:\d{2}$")

    print("\n--- 2단계: 텍스트 추출 및 파싱 결과 ---")

    for (y, x, text) in all_blocks:

        # 5-1. '나 (Me)'의 메시지 (오른쪽)
        if x > center_x:
            current_speaker = "나 (Me)"
            chat_log.append(f"{current_timestamp}, {current_speaker} : {text}")
            continue

        # 5-2. '왼쪽' 블록 분석

        # [수정된 핵심 로직]
        is_outside_main_column = (abs(x - main_text_column_x) > column_threshold)

        # 분기 1: 타임스탬프 열 (X좌표가 메인 열에서 크게 벗어남)
        if is_outside_main_column:
            if time_regex.match(text):
                current_timestamp = text
            # (시스템 메시지 등도 '턴 종료' 신호로 간주, 발화자는 초기화하지 않음)
            continue

        # 분기 2: 메인 열 (X좌표가 경계 근처에 있음)
        else:
            # [수정] X좌표가 중앙값(경계)보다 왼쪽에 있으면 '이름'
            if x < main_text_column_x:
                current_speaker = text # 새 발화자 이름으로 지정

            # [수정] X좌표가 중앙값(경계)보다 오른쪽에 있으면 '메시지'
            else:
                # '나'였다가 상대방으로 바뀐 직후 첫 메시지
                if current_speaker == "나 (Me)":
                     # '나' 다음의 첫 메시지는 이름이 없으므로, 이 메시지를 이름으로 잘못 판단할 수 있음
                     # 이 경우를 대비해, '나' 다음의 첫 텍스트는 이름으로 간주
                     if x < main_text_column_x:
                         current_speaker = text
                     else:
                         # '나' 다음에 바로 말풍선이 온 경우 (이름 없는 톡방 등)
                         # 이 부분은 정책이 필요함. 여기서는 일단 현재 스피커의 메시지로 처리
                         chat_log.append(f"{current_timestamp}, {current_speaker} : {text}")

                # 이미 상대방 발화자가 지정된 상태의 메시지
                else:
                    chat_log.append(f"{current_timestamp}, {current_speaker} : {text}")

    # 6. 최종 결과 출력
    for line in chat_log:
        print(line)

# --- 실행 ---
try:
    reader = easyocr.Reader(['ko', 'en'], gpu=True)
    print("EasyOCR 리더기 로드 완료.")

    image_file = "image.jpeg" # (X좌표 구분이 명확한 이미지)
    parse_chat_by_x_boundary(image_file, reader)

    print("\n" + "="*30 + "\n")

    image_file = "image.png"     # (X좌표 구분이 모호할 수 있는 이미지)
    parse_chat_by_x_boundary(image_file, reader)

except Exception as e:
    print(f"오류 발생: {e}")


In [ ]:
import easyocr
import re
from PIL import Image
import numpy as np

def parse_chat_by_x_column(image_path, reader):
    """
    X좌표(들여쓰기)를 분석하여 '메인 텍스트 열'과 '타임스탬프 열'을
    구분하고, 이를 기반으로 대화를 파싱합니다.
    (상단 필터링 로직 수정)
    """

    # 0. 이미지 크기 및 중앙선 계산
    try:
        pil_img = Image.open(image_path)
        img_width, img_height = pil_img.size # <-- 수정 1: img_height 가져오기
        center_x = img_width / 2
        print(f"--- 이미지 너비: {img_width}px, 높이: {img_height}px, 중앙 X: {center_x:.0f}px ---")
    except Exception as e:
        print(f"이미지 크기 읽기 실패: {e}")
        return

    # 1. OCR 실행 (좌표 포함)
    results = reader.readtext(image_path, paragraph=False)

    # 2. 텍스트 블록 정렬 및 처리 (왼쪽/오른쪽 구분)
    left_blocks = []  # (y_top, x_left, text)
    right_blocks = [] # (y_top, x_left, text)

    # 상단 상태바 필터링 기준 (전체 높이의 5%)
    top_filter_threshold = img_height * 0.05

    for (bbox, text, prob) in results:
        y_top = bbox[0][1]
        x_left = bbox[0][0]
        x_center = (bbox[0][0] + bbox[1][0]) / 2

        # --- 수정 2: 필터링 로직 변경 ---
        # 1. 상단 상태바 영역(상위 5%) 무시
        if y_top < top_filter_threshold:
            continue
        # 2. 안 읽은 수 무시
        if len(text) < 3 and text.isdigit():
            continue


        if x_center > center_x:
            right_blocks.append((y_top, x_left, text))
        else:
            left_blocks.append((y_top, x_left, text))

    # 3. [핵심 로직] 왼쪽 블록의 X좌표 통계 분석
    if not left_blocks:
        print("왼쪽 텍스트 블록을 찾을 수 없습니다.")
        return

    x_coords = [x for (y, x, text) in left_blocks]
    main_text_column_x = np.median(x_coords)
    column_threshold = img_width * 0.1

    print(f"--- 1단계: 메인 텍스트 열(x) 중앙값: {main_text_column_x:.0f}px ---")

    # 4. 모든 블록을 y_top 기준으로 재정렬
    all_blocks = left_blocks + right_blocks
    all_blocks.sort(key=lambda b: b[0])

    # 5. 정렬된 블록을 순회하며 대화 재구성
    chat_log = []
    current_speaker = None

    print("\n--- 2단계: 텍스트 추출 및 파싱 결과 ---")

    for (y, x, text) in all_blocks:

        # 5-1. '나 (Me)'의 메시지 (오른쪽)
        if x > center_x:
            chat_log.append(f"[나 (Me)]: {text}")
            current_speaker = "나 (Me)"
            continue

        # 5-2. '왼쪽' 블록 분석
        is_outside_main_column = (abs(x - main_text_column_x) > column_threshold)

        if is_outside_main_column:
            # 방 제목, 시스템 메시지("친구로 등록~"), 타임스탬프("더근") 등
            # 메인 열을 벗어난 텍스트는 '턴 종료' 신호로 간주

            # (선택적) 시스템 메시지 등은 출력
            if len(text) > 15: # "친구로 등록되지 않은~" 같이 긴 텍스트
                 chat_log.append(f"\n[시스템]: {text}")

            current_speaker = None # 발화자 초기화
            continue

        else:
            # '메인 열'에 속한 텍스트 (이름 또는 메시지)
            if current_speaker is None:
                current_speaker = text
                chat_log.append(f"\n--- {current_speaker} ---")
            else:
                if current_speaker == "나 (Me)":
                    current_speaker = text
                    chat_log.append(f"\n--- {current_speaker} ---")
                else:
                    chat_log.append(f"[{current_speaker}]: {text}")

    # 6. 최종 결과 출력

    for line in chat_log:
        print(line)

# --- 실행 ---
try:
    reader = easyocr.Reader(['ko', 'en'], gpu=True)
    print("EasyOCR 리더기 로드 완료.")

    image_file = "image.jpeg" # (첫 발화자가 높이 있는 이미지)
    parse_chat_by_x_column(image_file, reader)

    print("\n" + "="*30 + "\n")

    image_file = "image3.jpg"     # (오류가 났던 첫 번째 이미지)
    parse_chat_by_x_column(image_file, reader)

except Exception as e:
    print(f"오류 발생: {e}")
